# Создание словаря русского языка на основе данных Wiktionary

## Введение

В задачах обработки естественного языка (NLP) часто требуется словарь русского языка, содержащий полный список русских слов с правильной орфографией.

Такой словарь можно получить на основе данных сайта Wiktionary (Викисловарь).

Wiktionary - это открытый многоязычный словарь, в котором каждая словарная единица представлена в виде отдельной статьи. Викисловарь содержит информацию о словах различных языков, включая русский, а также сведения об их значениях, грамматических характеристиках и формах.

Для работы с данными проектов Wikimedia предоставляются регулярные дампы, публикуемые на https://dumps.wikimedia.org/.<br>
Дамп представляет собой архив с полной или частичной выгрузкой содержимого сайта в структурированном виде, пригодном для автоматизированной обработки.

## Создание словаря на основе названий статей

Скачаем файл `ruwiktionary-latest-all-titles-in-ns0.gz` по ссылке https://dumps.wikimedia.org/ruwiktionary/latest/ruwiktionary-latest-all-titles-in-ns0.gz.<br>
Файл содержит список заголовков всех статей из основного пространства имён русского раздела Wiktionary.

Выберем только статьи, названия которых состоят из букв русского алфавита, дефисов и апострофов (для слов типа д'артаньян).<br>
Список из таких названий статей как раз и будет словарем русского языка.

In [1]:
import gzip
import re
from pathlib import Path
import pandas as pd
import bz2
import shutil
import xml.etree.ElementTree as ET

In [2]:
input_file = Path("ruwiktionary-latest-all-titles-in-ns0.gz")
output_csv = Path("ruwiktionary_russian_words.csv")
output_txt = Path("ruwiktionary_russian_words.txt")

In [3]:
# Русское слово: кириллица, внутри допускаются дефисы и апострофы
ru_pattern = re.compile(r"^[А-Яа-яЁё]+(?:[-'][А-Яа-яЁё]+)*$")

words = set()

with gzip.open(input_file, "rt", encoding="utf-8") as file:
    for line in file:
        word = line.strip()

        # Оставляем только русские слова
        if ru_pattern.fullmatch(word):
            words.add(word.lower())

words = sorted(words)

df = pd.DataFrame(words, columns=["word"])

df.to_csv(output_csv, index=False, encoding="utf-8-sig")

with open(output_txt, "w", encoding="utf-8") as file:
    for word in words:
        file.write(word + "\n")

print(f"Готово. Найдено слов: {len(words)}")
print(f"CSV сохранён: {output_csv}")
print(f"TXT сохранён: {output_txt}")

Готово. Найдено слов: 1542954
CSV сохранён: ruwiktionary_russian_words.csv
TXT сохранён: ruwiktionary_russian_words.txt


Выведем первые строки словаря.

In [4]:
df.head(10)

,word
0,а
1,а'асх
2,а'атчевматальын
3,а'атчегматальын
4,а'к'алтывагыргын
5,а-а
6,а-а-а
7,а-а-а-а
8,а-ах
9,а-во


В таблице есть ненастоящие слова. Видимо в выборку попадают все статьи с кириллицей, а не только на русском языке. Также в выборке, возможно, есть ошибочные написания слов, страницы которых переводят на реальные слова. 

Для повышения качества словаря попробуем другой, более полный дамп, где будет возможность дополнительной фильтрации слов.

## Создание словаря на основе полного дампа содержимого русского Викисловаря

Скачаем файл `ruwiktionary-latest-pages-articles-multistream.xml.bz2` по ссылке https://dumps.wikimedia.org/ruwiktionary/latest/ruwiktionary-latest-pages-articles-multistream.xml.bz2.<br>
Файл содержит тексты всех статей русского раздела Wiktionary. 

Wiktionary является многоязычным ресурсом, одна статья может содержать разделы для нескольких языков. Для выделения русскоязычной части статьи используется wiki-шаблон {{-ru-}}, который обозначает начало раздела русского языка внутри статьи.

Фильтрация с помощью {{-ru-}} позволит отбросить пустые страницы и страницы, написанные на кириллице, но не имеющие русскоязычного раздела.

Распакуем архив.

In [5]:
input_bz2 = Path("ruwiktionary-latest-pages-articles-multistream.xml.bz2")
output_xml = Path("ruwiktionary-latest-pages-articles-multistream.xml")

with bz2.open(input_bz2, "rb") as src, open(output_xml, "wb") as dst:
    shutil.copyfileobj(src, dst, length=1024 * 1024 * 16)

print("Распаковка завершена")

Распаковка завершена


Выберем статьи с русским разделом.

In [6]:
input_xml = Path("ruwiktionary-latest-pages-articles-multistream.xml")
output_txt = Path("russian_dictionary.txt")

ru_pattern = re.compile(r"^[А-Яа-яЁё]+(?:[-'][А-Яа-яЁё]+)*$")

bad_patterns = [
    re.compile(r"^(?![авикосуя]$)[а-яё]$"),        # убираем односимвольные слова кроме а, в, и, к, о, с, у, я
    re.compile(r"^(а|о|у|э|и|ы|е|я|ё|ю)(-\1)+$"),  # убираем шум, крики, припевки типа а-а-а
]

words = set()
page_count = 0

with open(input_xml, "r", encoding="utf-8") as f:
    context = ET.iterparse(f, events=("end",))

    for event, elem in context:
        if elem.tag.endswith("page"):
            page_count += 1

            title = elem.find("./{*}title")
            text = elem.find(".//{*}text")

            if title is not None and text is not None:
                word = (title.text or "").strip().lower()
                article_text = text.text or ""

                if "{{-ru-}}" in article_text:
                    if (
                        ru_pattern.fullmatch(word)
                        and not any(p.fullmatch(word) for p in bad_patterns)
                    ):
                        words.add(word)

            if page_count % 100000 == 0:
                print(f"Обработано страниц: {page_count:,}, найдено слов: {len(words):,}")

            elem.clear()

with open(output_txt, "w", encoding="utf-8") as f:
    f.write("\n".join(sorted(words)))

print(f"Готово. Найдено слов: {len(words):,}")

Обработано страниц: 100,000, найдено слов: 5,421
Обработано страниц: 200,000, найдено слов: 57,430
Обработано страниц: 300,000, найдено слов: 122,536
Обработано страниц: 400,000, найдено слов: 124,940
Обработано страниц: 500,000, найдено слов: 125,854
Обработано страниц: 600,000, найдено слов: 149,921
Обработано страниц: 700,000, найдено слов: 156,096
Обработано страниц: 800,000, найдено слов: 162,552
Обработано страниц: 900,000, найдено слов: 205,546
Обработано страниц: 1,000,000, найдено слов: 215,474
Обработано страниц: 1,100,000, найдено слов: 249,555
Обработано страниц: 1,200,000, найдено слов: 293,223
Обработано страниц: 1,300,000, найдено слов: 323,535
Обработано страниц: 1,400,000, найдено слов: 405,248
Обработано страниц: 1,500,000, найдено слов: 414,217
Обработано страниц: 1,600,000, найдено слов: 414,505
Обработано страниц: 1,700,000, найдено слов: 415,578
Обработано страниц: 1,800,000, найдено слов: 415,683
Обработано страниц: 1,900,000, найдено слов: 415,734
Обработано стр

Сохраним в CSV

In [7]:
input_txt = Path("russian_dictionary.txt")
output_csv = Path("russian_dictionary.csv")

with open(input_txt, "r", encoding="utf-8") as f:
    words = [line.strip() for line in f if line.strip()]

df = pd.DataFrame(words, columns=["word"])
df.to_csv(output_csv, index=False, encoding="utf-8-sig")

print('Размер словаря:', len(df), 'слов')

Размер словаря: 444148 слов


Выведем первые строки словаря.

In [8]:
df.head(10)

,word
0,а
1,а-во
2,а-дато
3,а-каприччио
4,а-конто
5,а-ля
6,а-ля-карт
7,а-мольный
8,а-форфе
9,аа


Выведем случайные строки словаря.

In [9]:
df.sample(10, random_state=42)

,word
163564,лемго
231463,орзали
23402,бацкино
400904,укиха
235468,ответчица
238978,отнимавши
206539,неоязычество
439786,энрике
377324,сторновав
233787,остекление


Составим таблицу со значениями выбранных слов и оценим качество и пригодность словаря.

| Слово               | Значение                                                                   | Ссылка                                                                                                   |
| ------------------- | -------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------- |
| а                   | союз; противопоставление, сопоставление		                           | [https://ru.wiktionary.org/wiki/а](https://ru.wiktionary.org/wiki/а)                               |
| а-во                | сокращение от агентство			                           | [https://ru.wiktionary.org/wiki/а-во](https://ru.wiktionary.org/wiki/а-во)                               |
| а-дато              | финансовый термин: число, от которого дан тот или иной документ     	   | [https://ru.wiktionary.org/wiki/а-дато](https://ru.wiktionary.org/wiki/а-дато)                           |
| а-каприччио         | музыкальный термин: произвольно, не подчиняясь установившимся формам       | [https://ru.wiktionary.org/wiki/а-каприччио](https://ru.wiktionary.org/wiki/а-каприччио)                 |
| а-конто             | финансовый термин: вид предварительного расчёта, представляющий собой авансовую оплату счетов экспортёра импортёром за проданные товары  | [https://ru.wiktionary.org/wiki/а-конто](https://ru.wiktionary.org/wiki/а-конто)                         |
| а-ля                | подобно, наподобие, словно, по образцу                                                        | [https://ru.wiktionary.org/wiki/а-ля](https://ru.wiktionary.org/wiki/а-ля)                               |
| а-ля-карт    | по меню, по карте                                                             | [https://ru.wiktionary.org/wiki/а-ля-карт](https://ru.wiktionary.org/wiki/а-ля-карт)       |
| а-мольный           | написанный в тональности ля минор                                          | [https://ru.wiktionary.org/wiki/а-мольный](https://ru.wiktionary.org/wiki/а-мольный)                     |
| а-форфе             | заранее произведённая налоговыми органами оценка прибыли предприятия, на котором не ведётся точный бухгалтерский учёт  | [https://ru.wiktionary.org/wiki/а-форфе](https://ru.wiktionary.org/wiki/а-форфе)  |
| аа                  | междометие (эмоциональный звук)                                            | [https://ru.wiktionary.org/wiki/аа](https://ru.wiktionary.org/wiki/аа)                                   |
| лемго        | город в Германии                                              | [https://ru.wiktionary.org/wiki/Лемго](https://ru.wiktionary.org/wiki/Лемго)               |
| орзали       | мужское имя                                                                   | [https://ru.wiktionary.org/wiki/Орзали](https://ru.wiktionary.org/wiki/Орзали)             |
| бацкино      | село в России                                                     | [https://ru.wiktionary.org/wiki/Бацкино](https://ru.wiktionary.org/wiki/Бацкино)           |
| укиха        | город в Японии                                                       | [https://ru.wiktionary.org/wiki/Укиха](https://ru.wiktionary.org/wiki/Укиха)               |
| ответчица    | женск. к ответчик                                                        | [https://ru.wiktionary.org/wiki/ответчица](https://ru.wiktionary.org/wiki/ответчица)       |
| отнимавши    | дееприч. от отнимать                                                     | [https://ru.wiktionary.org/wiki/отнимавши](https://ru.wiktionary.org/wiki/отнимавши)       |
| неоязычество | новое язычество; движение, связанное с возрождением доавраамических верований | [https://ru.wiktionary.org/wiki/неоязычество](https://ru.wiktionary.org/wiki/неоязычество) |
| энрике       | мужское имя                                                                   | [https://ru.wiktionary.org/wiki/Энрике](https://ru.wiktionary.org/wiki/Энрике)             |
| сторновав    | дееприч. от сторновать                                                       | [https://ru.wiktionary.org/wiki/сторновав](https://ru.wiktionary.org/wiki/сторновав)       |
| остекление   | действие по значению гл. остеклить                                      | [https://ru.wiktionary.org/wiki/остекление](https://ru.wiktionary.org/wiki/остекление)     |



Из таблицы видно, что большинство слов действительно имеют значение и употребляются в русскоязычных текстах. В словаре встречаются междометия, которые не несут большого смыслового значения, но тем не менее употребляются в речи. Также в словаре есть топонимы, имена и фамилии, много редко используемых, устаревших или узкоспециализированных слов, что естественно ввиду большого размера словаря.

Важно отметить, что полученный словарь не лемматизирован и включает как леммы, так и различные словоформы (например, причастия и деепричастия).

## Вывод

Для точных задач целесообразно использовать дополнительные методы фильтрации или альтернативные источники словарей.<br>
Полученный словарь может быть использован для простых задач NLP.